# Titanic Survival Prediction

This version is tuned for the Kaggle public score after the previous overfit ensemble scored `0.77272`.

Changes in this version:
- Adds leak-safe surname and ticket survival features.
- Evaluates those features with fold-by-fold encoding, so validation rows do not see their own labels.
- Creates several candidate submissions instead of only one overfit ensemble.
- Uses the best local CV candidate, `gb_group`, as `submission.csv`.


In [1]:
import pandas as pd

from titanic_candidates import cross_validate_models, make_models, write_submissions


In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
y = train_df["Survived"].astype(int)

print(train_df.shape)
print(test_df.shape)
train_df.head()


(891, 12)
(418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Cross-Validation

The helper script computes surname/ticket survival features inside each fold. This is important because normal target encoding can leak labels and make validation scores too optimistic.


In [3]:
models = make_models()
cv_results = cross_validate_models(train_df, y, models)
cv_results


,model,mean_accuracy,std
0,gb_group,0.848452,0.023333
3,vote_group,0.848427,0.028293
1,rf_group,0.842809,0.027753
2,et_group,0.840587,0.021436


## Create Candidate Submissions

`submission.csv` is copied from `submission_gb_group.csv`, the best local 10-fold CV model. The other files are useful Kaggle backup submissions because public leaderboard behavior can differ from local CV on this small dataset.


In [4]:
write_submissions(train_df, test_df, y, models)


submission_gb_group.csv
Survived
0    264
1    154
Name: count, dtype: int64


submission_rf_group.csv
Survived
0    273
1    145
Name: count, dtype: int64


submission_et_group.csv
Survived
0    271
1    147
Name: count, dtype: int64


submission_vote_group.csv
Survived
0    271
1    147
Name: count, dtype: int64
submission.csv updated from submission_gb_group.csv


In [5]:
submission = pd.read_csv("submission.csv")
print(submission.shape)
print(submission.dtypes)
print(submission["Survived"].value_counts().sort_index())
submission.head()


(418, 2)
PassengerId    int64
Survived       int64
dtype: object
Survived
0    264
1    154
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
